In [ ]:
# Initialize Otter
import otter
grader = otter.Notebook("hwk3-task2-spills.ipynb")

# Task 2: Visualizing oil spills in NY State

## Instructions

- First, update the following cell to have a link to *your* Homework 3 GitHub repository:

https://github.com/fatiqnadeem/eds220-hwk3


- Review the [complete rubric for this task](https://docs.google.com/document/d/1Ce4EO3MEHW5uLks4x8BECWlkim2bcmxB4f8Qo4z26z8/edit?usp=sharing) before starting.

- **Meaningful commits should be made every time you finish a major step.** We'll check your repository and view the commit history.

- Comment mindfully in a way that enriches your code. Comments should follow best practices.

- **Do not update the top cell with the `otter` import**, this is used internally for grading.


## About the data
In this task you will use two datasets:

### First dataset: oil spills data

The first dataset contains information about [spilling incidents in the state of New York](https://data.ny.gov/Energy-Environment/Spill-Incidents/u44d-k5fk). Make sure you study the data portal and understand what each row represents before starting to code. 
You will find extra information and metadata for this dataset in the `NYSDEC_SpillIncidents_Overview.pdf` and `NYSDEC_SpillIncidents_DataDictionary.pdf` documents available at the portal.

### Second dataset: TIGER shapefile

The second dataset is the [2026 Counties (and equivalent) TIGER shapefile](https://www.census.gov/geographies/mapping-files/time-series/geo/tiger-line-file.2026.html) (file `tl_2026_us_county.zip`) from the Census Bureau. You can check the [metadata for all the TIGER shapefiles here](https://www.census.gov/programs-surveys/geography/technical-documentation/complete-technical-documentation/tiger-geo-line.html). 

### COMPLETE WORKFLOW
You will use the next code cell to complete the last exercise in the task. Leave it blank for now.

In [ ]:
# FINAL CODE

## 1. File management
a. Any data you download should go in the `data/` folder of your `eds220-hwk3` repository. For example, after downloading both datasets, your repository should look like this:

```
eds220-hwk3/
├── data/
│   ├── q5d_df.csv
│   ├── Spill_Incidents_20261027.csv
│   └── tl_2026_us_county/
│       ├── tl_2026_us_county.cpg
│       ├── tl_2026_us_county.dbf
│       ├── tl_2026_us_county.prj
│       ├── tl_2026_us_county.shp
│       └── ... (other files from the shapefile)
├── tests/
├── .gitignore
├── LICENSE
├── README.md
└── hwk3-task2-spills.ipynb
```

The date in the name of the spills file may be different, depending on when you download it. `data/q5d_df.csv` is already in the repository and is used by the autograder tests, so do not delete or move it.

**IMPORTANT: Do not commit your data!**

The datasets we'll be working with are large and will most likely jam your commit history if git does not ignore them.

b. Add the relative path of each dataset you downloaded to your `.gitignore` file. For example, to ignore the folder containing the shapefile, add `data/tl_2026_us_county/` to the `.gitignore`.

c. Commit and push your changes to the `.gitignore`.

d. Run `git status` and verify that none of your data is being tracked by git.

## 2. Datasets description
- Read the metadata for both datasets and write a brief description about each one. Indicate the data sources and date of access.

**Spill Incidents (New York State Department of Environmental Conservation).**
The first dataset is the NYSDEC Spill Incidents table published on the New York State open data portal
([data.ny.gov, dataset u44d-k5fk](https://data.ny.gov/Energy-Environment/Spill-Incidents/u44d-k5fk)).
It records petroleum, hazardous material and other spills reported to the Department since the 1970s and is
updated daily. Each row holds one material reported for one spill incident, and the incident is identified by its
spill number, so an incident in which several materials were released occupies several rows. The 20 columns describe
the incident (spill number, program facility name, street, locality, county, ZIP code, SWIS code, DEC region),
its timing (spill date, received date, close date), its circumstances (contributing factor, waterbody, source), and the
material (material name, material family, quantity, units, recovered). The file I downloaded as a CSV export on
October 30, 2026 has 574,628 rows.

**TIGER/Line 2026 Counties and Equivalent Entities (U.S. Census Bureau).**
The second dataset is the 2026 release of the Census Bureau's TIGER/Line county shapefile
(`tl_2026_us_county.zip`, downloaded from the
[TIGER/Line 2026 page](https://www.census.gov/geographies/mapping-files/time-series/geo/tiger-line-file.2026.html)
on October 30, 2026). It holds one polygon for each of the 3,235 counties and county equivalents in the United States,
Puerto Rico and the Island Areas, with 18 attribute columns including the state and county FIPS codes (`STATEFP`,
`COUNTYFP`, `GEOID`), the county name (`NAME`) and its name with legal description (`NAMELSAD`), land and water area
(`ALAND`, `AWATER`) and an interior point. The geometries are in the NAD83 geographic coordinate system (EPSG:4269).

## 3. Import libraries

In [ ]:
# Libraries for file paths, numerics, tabular data, plotting and vector geodata
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import geopandas as gpd

## 4. Import data 
In the next cell:

- Import the NY spills data as a variable named `spills`. 
- Import the US counties data as a variable named `counties`.

If you are using file paths to access the data, these should be constructed using the `os` package.

In [ ]:
# Spill numbers are identifiers with leading zeros, so read them as text
fp = os.path.join('data', 'Spill_Incidents_20261030.csv')
spills = pd.read_csv(fp, dtype={'Spill Number': str})

# TIGER/Line 2026 counties shapefile
fp = os.path.join('data', 'tl_2026_us_county', 'tl_2026_us_county.shp')
counties = gpd.read_file(fp)

<!-- BEGIN QUESTION -->

## 5. Prepare spills data

This section refers to the `spills` dataframe. 
Check the outputs carefully, they will give you information you will need to use in the next exercises.

### 5.A. Exploration

In the next cells:

a. Check the dataframe's head

b. Check the data types of the columns

c. Check the unique values in the `material_family` column

d. Check the unique values in the `county` column

e. Insert a markdown cell to explain what information you obtained from the preliminary exploration. Use complete sentences.

You can add any other cells for preliminary data exploration.

In [ ]:
# a. First rows of the spills data
spills.head()

In [ ]:
# b. Data type of each column
spills.dtypes

In [ ]:
# c. Material families recorded (column names are still the raw ones with capitals and spaces)
spills['Material Family'].unique()

In [ ]:
# d. Every distinct value of the county column
spills['County'].unique()

In [ ]:
# Dimensions of the full dataset and the number of distinct county labels
print(f"Rows and columns: {spills.shape}")
print(f"Distinct values in County: {spills['County'].nunique()}")

The spills table has 574,628 rows and 20 columns. The column names mix capital letters and spaces
(for example `Spill Number` and `Material Family`), so I will simplify them before working with the data.
`pandas` read the three date columns (`Spill Date`, `Received Date`, `Close Date`) as text in the form
`MM/DD/YYYY`, so they need to be converted to datetime before I can select a year. The `Material Family` column
takes only four values (Petroleum, Hazardous Material, Other and Oxygenates), which means the petroleum filter
in the next section is a simple equality test. The `County` column has 87 distinct labels, and they are not all
New York counties. It also contains DEC regions in neighbouring states and Canada (for example
`New Jersey - Region 2`), several Indian Reservations listed as their own jurisdiction, and `St Lawrence`
spelled without a period, all of which will matter when I match these names to the Census county names.

<!-- END QUESTION -->

<!-- BEGIN QUESTION -->

### 5.B. Cleaning
a. Simplify column names as needed.

In [ ]:
# Lower-case the column names and replace spaces with underscores
spills.columns = spills.columns.str.lower().str.replace(' ', '_')
spills.columns

<!-- END QUESTION -->

<!-- BEGIN QUESTION -->

b. Update column data types as needed.

In [ ]:
# Convert the three date columns from MM/DD/YYYY text to datetime
for col in ['spill_date', 'received_date', 'close_date']:
    spills[col] = pd.to_datetime(spills[col], format='%m/%d/%Y')

spills.dtypes

<!-- END QUESTION -->

<!-- BEGIN QUESTION -->

### 5.C. Data selection

a. Select data about petroleum spills that took place between January 1 and December 31, 2023 (including both dates). Store it in a variable named `petr_23`. The data selection should occur in a single line. You may add cells as needed.

In [ ]:
# Petroleum spills whose spill date falls in calendar year 2023 (between() is inclusive at both ends)
petr_23 = spills[(spills['material_family'] == 'Petroleum') & (spills['spill_date'].between('2023-01-01', '2023-12-31'))]

<!-- END QUESTION -->

<!-- BEGIN QUESTION -->

b. Use `print()` and f-strings to print a message verifying that `petr_23` only has data for 2023 and from January through December.
HINT: if you converted the `spill_date` column to datetime, you can use `petr_23['spill_date'].dt.year.unique()`.

In [ ]:
# Confirm the selection covers 2023 only and every month of that year
print(f"Years present in petr_23: {petr_23['spill_date'].dt.year.unique().tolist()}")
print(f"Months present in petr_23: {sorted(petr_23['spill_date'].dt.month.unique().tolist())}")
print(f"Earliest spill date: {petr_23['spill_date'].min().date()}, latest spill date: {petr_23['spill_date'].max().date()}")

<!-- END QUESTION -->

<!-- BEGIN QUESTION -->

c. Write tests using `assert` to verify that `petr_23` only has data for petroleum spills.

In [ ]:
# Every row must be petroleum, and petroleum must be the only material family present
assert (petr_23['material_family'] == 'Petroleum').all()
assert petr_23['material_family'].nunique() == 1
assert 'Petroleum' in petr_23['material_family'].unique()
print("All assert tests passed: petr_23 contains petroleum spills only.")

<!-- END QUESTION -->

<!-- BEGIN QUESTION -->

d. Use `print()` and f-strings to print the number of rows in `petr_23` and the number of unique spills (use the `spill_number` column). Are these numbers the same? In the next markdown cell, explain why this might be the case. Use complete sentences. You may use additional cells to explore the data if needed.

In [ ]:
# Rows versus distinct spill numbers
print(f"Number of rows in petr_23: {len(petr_23)}")
print(f"Number of unique spills in petr_23: {petr_23['spill_number'].nunique()}")

In [ ]:
# Look at one spill number that appears in more than one row
(petr_23[petr_23['spill_number'].duplicated(keep=False)]
     .sort_values('spill_number')
     [['spill_number', 'spill_date', 'county', 'material_name', 'quantity', 'units']]
     .head(6))

The two numbers differ. `petr_23` has 8,468 rows but only 8,228 distinct spill numbers. A row in this
dataset describes one material reported for one incident rather than the incident itself, so a spill in
which several petroleum products were released, for example gasoline and diesel escaping from the same tank,
appears once per material under the same spill number. The table above shows this pattern, where consecutive
rows share a spill number, date and county and differ only in the material name. Counting rows would therefore
overstate the number of spills by about three percent, and in the next exercise I count distinct spill numbers
instead.

<!-- END QUESTION -->

### 5.D. Data aggregation

Create new dataframe named `spills_per_county` with the number of petroleum spills per county from January to December 2023.

As you found in exercise 5.C.d, a spill can appear in more than one row. Make sure you count each spill only once. HINT: `nunique()`.

The resulting `spills_per_county` dataframe must be as follows:

- Index: integer numbers starting from 0
- Two columns only.
- Column one: county names, column name = `county`
- Column two: number of petroleum spills per county in 2023, column name = `n_spills`

You may add cells if needed for your data wrangling.

In [ ]:
# Distinct spill numbers per county, returned as a two-column dataframe with a fresh integer index
spills_per_county = (petr_23.groupby('county')['spill_number']
                            .nunique()
                            .reset_index(name='n_spills'))
spills_per_county

In [ ]:
grader.check("q5d")

<!-- BEGIN QUESTION -->

## 6. Prepare counties data

This section refers to the `counties` geo-dataframe. 
Check the outputs carefully, they will give you context about the next exercises.

### 6.A. Exploration

In the next cells:

a. Check the geo-dataframe's head

b. Check the data types of the columns

c. Check the geo-dataframe's CRS

d. Plot the geo-dataframe

e. Use the next markdown cell to explain what information you obtained from the preliminary exploration. Use complete sentences.

You can add any other cells of preliminary data exploration.

In [ ]:
# a. First rows of the counties geo-dataframe
counties.head()

In [ ]:
# b. Data type of each column
counties.dtypes

In [ ]:
# c. Coordinate reference system
counties.crs

In [ ]:
# d. Quick look at every polygon in the file
counties.plot()

The counties file is a `GeoDataFrame` with 3,235 rows and 19 columns, one row per county or county
equivalent. The identifier columns (`STATEFP`, `COUNTYFP`, `GEOID`) are stored as text, which preserves
their leading zeros, so selecting a state means comparing against a string such as `'36'` rather than an
integer. `NAME` holds the plain county name and `NAMELSAD` adds the legal description (for example
"Albany County"), while `ALAND` and `AWATER` are integer areas in square metres. The CRS is EPSG:4269,
the NAD83 geographic system in degrees of latitude and longitude, so the data are unprojected. The plot
shows the whole country together with Alaska, Hawaii, Puerto Rico and the Pacific islands, with Alaska's
Aleutian chain stretching across the antimeridian, which is why the map looks compressed and why I need to
select New York before plotting anything useful.

<!-- END QUESTION -->

<!-- BEGIN QUESTION -->

### 6.B. Cleaning

Simplify column names as needed.

In [ ]:
# Lower-case the column names
counties.columns = counties.columns.str.lower()
counties.columns

<!-- END QUESTION -->

<!-- BEGIN QUESTION -->

### 6.C. Data selection

Select all the counties for the state of New York in a new variable `ny_counties` and plot them (you may need to look at the metadata).

In [ ]:
# New York State has FIPS code 36; statefp is text, so compare with a string
ny_counties = counties[counties['statefp'] == '36']

fig, ax = plt.subplots(figsize=(7, 6))
ny_counties.plot(ax=ax, edgecolor='white', linewidth=0.5)
ax.set_title('Counties of New York State (TIGER/Line 2026)')
ax.set_axis_off()
plt.show()

<!-- END QUESTION -->

<!-- BEGIN QUESTION -->

## 7 Merging

The goal of this section is to create a single dataframe with the number of spills per county and the corresponding geometries for each county.

### 7.A. Explore data

In the next cells:

a. Print the names of counties in `ny_counties`, without repetition.


<!-- END QUESTION -->

<!-- BEGIN QUESTION -->

b. Print the names of counties in `spills_per_county`, without repetition. Examine the differences in the outputs.

<!-- END QUESTION -->



c. Which are the county names that are in `spills_per_county`, but not in `ny_counties`? Your answer should be a list of strings
HINT: You can use the `numpy` function [`setdiff1d()`](https://numpy.org/doc/stable/reference/generated/numpy.setdiff1d.html), pay attention to the example in the documentation. Store your answer in the `diff_names` variable. 


In [ ]:
diff_names_spills = ...
diff_names_spills


In [ ]:
grader.check("7Ac")

d. Which are the county names that are in `ny_counties`, but not in `spills_per_county`? Your answer should be a list of a string/strings.


In [ ]:
diff_names_ny = ...
diff_names_ny

In [ ]:
grader.check("7Ad")

e. What is the number of spills in 'Tuscarora Indian Reservation' and in 'Onondaga Indian Reservation'?


In [ ]:
# e. 

### 7.B. Data updates

In the next cell, update 'St Lawrence' to 'St. Lawrence' in the `spills_per_county` data frame. Verify the update was successful with an assert test. 


### 7.C. Merge

In the following cells:

a. Use this cell to make any other updates needed for making an inner join of the `spills_per_county` and `ny_counties`. HINT: [`pd.merge`](https://pandas.pydata.org/pandas-docs/version/2.3/reference/api/pandas.merge.html)


In [ ]:
# a 


b. Store the inner join of the `spills_per_county` and `ny_counties` dataframes in a new variable named `ny_spills`.


<!-- BEGIN QUESTION -->

c. An inner join only keeps the rows whose `county` appears in both dataframes, and it does not warn you about the rows it drops. Which records from `spills_per_county` were dropped by the inner join, and how many spills did each one have? Add the cells you need to find the dropped records.

<!-- END QUESTION -->

<!-- BEGIN QUESTION -->

d. Does dropping these records affect the map or the information we will report? Write your answer in the following markdown cell.

_Type your answer here, replacing this text._

<!-- END QUESTION -->


From our previous exploration we know that Tuscarora Indian Reservation and Onondaga Indian Reservation will not be in the join. Tuscarora Indian Reservation is located within Niagara county and Onondaga Indian Reservation is located within Onondaga county. 

e. Since the reservations do not appear in the `ny_counties` dataframe, add the Tuscarora Indian Reservation spills to Niagara county and the Onondaga Indian Reservation spills to Onondaga county in `ny_spills`. We will add a note about this in our final map, and we will reflect on this choice in exercise 8.b. Add the cells you need to make this update **and** verify that the dataframe was updated.

## 8. Map
a. Create a choropleth map of the number of petroleum spills per county in the state of New York in 2023. 
Your map should be polished and have (at least) the following updates:
- a legend indicating the number of spills
- an updated color map
- a title 
- no axis
- an annotation with the data source (New York State Department of Environmental Conservation) and date of access



b. Write a description of how the map was created, including key takeaways. If you need to do additional data analysis to include relevant information from the data, you may include additional cells here.

In your description, also reflect on how we handled the spills reported in the Tuscarora and Onondaga Indian Reservations (exercise 7.C.e) in terms of data governance and the CARE Principles for Indigenous Data Governance from Task 1.

## Complete workflow

Collect all the relevant code into the first blank cell of the notebook titled "COMPLETE WORKFLOW". This single cell will have the end-to-end workflow: from importing libraries and loading the data, to producing the choropleth map of the number of petroleum spills per county in the state of New York in 2023. The only output of this cell should be the choropleth map  you produced in the previous exercise. Further guidance on what to include in this final workflow is in the assignment rubric.